## inport

In [ ]:
import json
import os

## ディレクトリ設定

In [ ]:
# 入力ファイルと出力フォルダの指定
json_path = "yolox_annotations.json"
output_dir = "yolo_labels"
os.makedirs(output_dir, exist_ok=True)

## main

In [ ]:


# JSON読み込み
with open(json_path, 'r') as f:
    data = json.load(f)

# クラスIDと名前のマッピング作成
id_to_name = {cat["id"]: cat["name"] for cat in data["categories"]}
name_to_id = {name: i for i, name in enumerate(sorted(set(id_to_name.values())))}

# DATASETTING.jsonとして保存
with open("DATASETTING.json", 'w') as f:
    json.dump(name_to_id, f, indent=2)

# 画像ごとにアノテーションをYOLO形式で出力
images = {img["id"]: img for img in data["images"]}

for ann in data["annotations"]:
    image = images[ann["image_id"]]
    file_name = image["file_name"]
    width = image["width"]
    height = image["height"]

    # bbox変換: x, y, w, h → YOLO形式 (cx, cy, w, h)（正規化）
    x, y, w, h = ann["bbox"]
    cx = (x + w / 2) / width
    cy = (y + h / 2) / height
    w /= width
    h /= height

    # クラスIDをマッピングで置換（連番）
    class_name = id_to_name[ann["category_id"]]
    class_id = name_to_id[class_name]

    # 出力ファイルに追記
    txt_filename = os.path.splitext(file_name)[0] + ".txt"
    with open(os.path.join(output_dir, txt_filename), 'a') as f:
        f.write(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")